<a href="https://colab.research.google.com/github/Uyhuynh23/Group13_CV/blob/main/notebooks/colab_flickr8k_seed999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flickr8k 2×2 Experiment Runner — Seed 999
**Project:** *A Controlled Evaluation of Patch-Preserving and Query-Resampling Connectors for Resource-Constrained Image Captioning*  
**Group 13:** Huỳnh Trần Uy (23125047), Trần Mỹ An (23125049), Nguyễn Hoàng Gia Hân (23125055)

This notebook runs all **4 main experimental conditions (A, B, C, D)** with **`seed=999`** and saves all checkpoints and evaluation JSONs to the shared Google Drive folder (`/content/drive/MyDrive/Group13_CV_Data`).

## Step 1: Mount Google Drive & Verify Tesla T4 GPU

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Group13_CV_Data'
ZIP_CACHE_DIR = f'{DRIVE_ROOT}/flickr8k_zips'
CLIP_CACHE_DIR = f'{DRIVE_ROOT}/cache'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
LOCAL_DATA_DIR = '/content/flickr8k_local'
SEED = 999

for d in [DRIVE_ROOT, ZIP_CACHE_DIR, CLIP_CACHE_DIR, CKPT_DIR, RESULTS_DIR, LOCAL_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Persistent Google Drive root:', DRIVE_ROOT)
print('Experiment Seed:', SEED)
!nvidia-smi

Mounted at /content/drive
Persistent Google Drive root: /content/drive/MyDrive/Group13_CV_Data
Experiment Seed: 999
Wed Nov 18 02:30:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                       

## Step 2: Clone / Pull GitHub Repository & Install Dependencies

In [3]:
REPO_URL = 'https://github.com/Uyhuynh23/Group13_CV.git'
REPO_DIR = '/content/Group13_CV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

Cloning into '/content/Group13_CV'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 214 (delta 104), reused 103 (delta 81), pack-reused 76 (from 1)
Receiving objects: 100% (214/214), 2.31 MiB | 38.79 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/Group13_CV
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Step 3: Verify / Generate Cached CLIP ViT-B/32 Features
Skips automatically if `train_cache.pt`, `val_cache.pt`, and `test_cache.pt` already exist on Google Drive.

In [4]:
%cd /content/Group13_CV

expected_caches = [
    f'{CLIP_CACHE_DIR}/train_cache.pt',
    f'{CLIP_CACHE_DIR}/val_cache.pt',
    f'{CLIP_CACHE_DIR}/test_cache.pt',
]

if all(os.path.isfile(p) for p in expected_caches):
    print('[SKIP] All CLIP feature caches already exist on Google Drive!')
    for p in expected_caches:
        print(f'  - {p} ({os.path.getsize(p)/(1024**2):.1f} MB)')
else:
    !python download_flickr8k.py --data_dir {LOCAL_DATA_DIR} --zip_cache_dir {ZIP_CACHE_DIR}
    !python parse_flickr8k.py \
        --data_dir {LOCAL_DATA_DIR} \
        --splits_dir ./data/flickr8k/splits \
        --cache_dir {CLIP_CACHE_DIR} \
        --batch_size 64

/content/Group13_CV
[SKIP] All CLIP feature caches already exist on Google Drive!
  - /content/drive/MyDrive/Group13_CV_Data/cache/train_cache.pt (439.9 MB)
  - /content/drive/MyDrive/Group13_CV_Data/cache/val_cache.pt (73.3 MB)
  - /content/drive/MyDrive/Group13_CV_Data/cache/test_cache.pt (73.3 MB)


## Step 4: Condition A — Patch-Preserving MLP + Frozen GPT-2 (`patch_mlp`) (Seed 999)

In [4]:
# %cd /content/Group13_CV

# RUN_NAME = 'condA_patch_mlp_frozen_seed999'
# BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
# TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

# if os.path.isfile(BEST_CKPT):
#     print(f'[SKIP] Condition A checkpoint already exists: {BEST_CKPT}')
# else:
#     !python train_flickr8k.py \
#         --cache_dir {CLIP_CACHE_DIR} \
#         --out_dir {CKPT_DIR} \
#         --run_name {RUN_NAME} \
#         --connector_type patch_mlp \
#         --prefix_length 49 \
#         --epochs 10 \
#         --bs 64 \
#         --lr 2e-4 \
#         --patience 4 \
#         --seed {SEED}

# # Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
# !python evaluate.py \
#     --checkpoint {BEST_CKPT} \
#     --cache_path {TEST_CACHE} \
#     --out_dir {RESULTS_DIR} \
#     --beam_size 5 \
#     --max_new_tokens 35

# !python evaluate.py \
#     --checkpoint {BEST_CKPT} \
#     --cache_path {TEST_CACHE} \
#     --out_dir {RESULTS_DIR} \
#     --beam_size 1 \
#     --max_new_tokens 35

/content/Group13_CV
 TRAINING FLICKR8K MODEL: condA_patch_mlp_frozen_seed999
 Device: cuda | Mixed Precision (AMP): True | Seed: 999
 Connector: patch_mlp | Visual Prefix Length: 49 | LoRA: False
config.json: 100% 665/665 [00:00<00:00, 2.73MB/s]

model.safetensors: downloading bytes:  36% 199M/548M [00:01<00:01, 278MB/s, 12.1MB/s  ]
model.safetensors: downloading bytes:  50% 276M/548M [00:01<00:00, 383MB/s, 19.0MB/s  ]
model.safetensors: downloading bytes:  80% 438M/548M [00:01<00:00, 465MB/s, 35.3MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:02<00:00, 172MB/s, 43.3MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:02<00:00, 198MB/s, 51.0MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 6350.00it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 472kB/s]
 Trainable Parameters : 1,181,184 (Connector: 1,181,184, GPT-2 LoRA: 0)
 Frozen GPT-2 Params  : 124,439,808
 Total Parameters     : 125,620,992
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00

## Step 5: Condition B — Query Resampler + Frozen GPT-2 (`query_resampler`) (Seed 999)

In [5]:
# %cd /content/Group13_CV

# RUN_NAME = 'condB_query_resampler_frozen_seed999'
# BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
# TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

# if os.path.isfile(BEST_CKPT):
#     print(f'[SKIP] Condition B checkpoint already exists: {BEST_CKPT}')
# else:
#     !python train_flickr8k.py \
#         --cache_dir {CLIP_CACHE_DIR} \
#         --out_dir {CKPT_DIR} \
#         --run_name {RUN_NAME} \
#         --connector_type query_resampler \
#         --prefix_length 16 \
#         --epochs 10 \
#         --bs 64 \
#         --lr 2e-4 \
#         --patience 4 \
#         --seed {SEED}

# # Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
# !python evaluate.py \
#     --checkpoint {BEST_CKPT} \
#     --cache_path {TEST_CACHE} \
#     --out_dir {RESULTS_DIR} \
#     --beam_size 5 \
#     --max_new_tokens 35

# !python evaluate.py \
#     --checkpoint {BEST_CKPT} \
#     --cache_path {TEST_CACHE} \
#     --out_dir {RESULTS_DIR} \
#     --beam_size 1 \
#     --max_new_tokens 35

/content/Group13_CV
 TRAINING FLICKR8K MODEL: condB_query_resampler_frozen_seed999
 Device: cuda | Mixed Precision (AMP): True | Seed: 999
 Connector: query_resampler | Visual Prefix Length: 16 | LoRA: False
Loading weights: 100% 148/148 [00:00<00:00, 7003.68it/s]
 Trainable Parameters : 992,768 (Connector: 992,768, GPT-2 LoRA: 0)
 Frozen GPT-2 Params  : 124,439,808
 Total Parameters     : 125,432,576
Epoch 1/10:   0% 0/468 [00:00<?, ?it/s]/content/Group13_CV/train_flickr8k.py:239: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch 1/10: 100% 468/468 [01:25<00:00,  5.48it/s, loss=2.8377, lr=1.89e-04]
Val Epoch 1: 100% 1000/1000 

## Step 6: Condition C — Patch-Preserving MLP + LoRA GPT-2 (`patch_mlp` + LoRA) (Seed 999)

In [5]:
%cd /content/Group13_CV

RUN_NAME = 'condC_patch_mlp_lora_seed999'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

if os.path.isfile(BEST_CKPT):
    print(f'[SKIP] Condition C checkpoint already exists: {BEST_CKPT}')
else:
    !python train_flickr8k.py \
        --cache_dir {CLIP_CACHE_DIR} \
        --out_dir {CKPT_DIR} \
        --run_name {RUN_NAME} \
        --connector_type patch_mlp \
        --prefix_length 49 \
        --use_lora \
        --lora_r 8 \
        --lora_alpha 16 \
        --lora_dropout 0.05 \
        --epochs 10 \
        --bs 64 \
        --lr 2e-4 \
        --patience 4 \
        --seed {SEED}

# Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

/content/Group13_CV
[SKIP] Condition C checkpoint already exists: /content/drive/MyDrive/Group13_CV_Data/checkpoints/condC_patch_mlp_lora_seed999_best.pt
Loading checkpoint from /content/drive/MyDrive/Group13_CV_Data/checkpoints/condC_patch_mlp_lora_seed999_best.pt on cuda ...
config.json: 100% 665/665 [00:00<00:00, 2.61MB/s]

model.safetensors: downloading bytes:  37% 203M/548M [00:01<00:01, 202MB/s, 14.0MB/s  ]
model.safetensors: downloading bytes:  48% 263M/548M [00:01<00:01, 271MB/s, 19.3MB/s  ]
model.safetensors: downloading bytes:  77% 425M/548M [00:02<00:00, 398MB/s, 35.3MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:02<00:00, 162MB/s, 43.2MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:02<00:00, 187MB/s, 50.9MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 713.46it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 562kB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 131kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 1.90MB/s]

## Step 7: Condition D — Query Resampler + LoRA GPT-2 (`query_resampler` + LoRA) (Seed 999)

In [6]:
%cd /content/Group13_CV

RUN_NAME = 'condD_query_resampler_lora_seed999'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

if os.path.isfile(BEST_CKPT):
    print(f'[SKIP] Condition D checkpoint already exists: {BEST_CKPT}')
else:
    !python train_flickr8k.py \
        --cache_dir {CLIP_CACHE_DIR} \
        --out_dir {CKPT_DIR} \
        --run_name {RUN_NAME} \
        --connector_type query_resampler \
        --prefix_length 16 \
        --use_lora \
        --lora_r 8 \
        --lora_alpha 16 \
        --lora_dropout 0.05 \
        --epochs 10 \
        --bs 64 \
        --lr 2e-4 \
        --patience 4 \
        --seed {SEED}

# Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

/content/Group13_CV
[SKIP] Condition D checkpoint already exists: /content/drive/MyDrive/Group13_CV_Data/checkpoints/condD_query_resampler_lora_seed999_best.pt
Loading checkpoint from /content/drive/MyDrive/Group13_CV_Data/checkpoints/condD_query_resampler_lora_seed999_best.pt on cuda ...
Loading weights: 100% 148/148 [00:00<00:00, 5409.74it/s]
Evaluating test (beam=5): 100% 1000/1000 [03:28<00:00,  4.79it/s]
PTBTokenizer tokenized 64178 tokens at 434024.61 tokens per second.
PTBTokenizer tokenized 12260 tokens at 141537.36 tokens per second.
{'testlen': 10268, 'reflen': 10001, 'guess': [10268, 9268, 8268, 7268], 'correct': [4624, 980, 186, 74]}
ratio: 1.0266973302668705
[WARN] Java METEOR failed (could not convert string to float: b'12.0 11.0 6.0 5.0 1.0 1.0 3.0 3.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 3.0 4.0 4.0'); falling back to NLTK METEOR.

 EVALUATION RESULTS (TEST SPLIT | query_resampler | beam5)
  CIDEr                    : 5.87
  BLEU-1                   : 45.03
 

## Step 8: Summary Table for Seed 999 (Conditions A, B, C, D)

In [7]:
import json
import os

runs = [
    ('Cond A: patch_mlp + Frozen', 'eval_test_condA_patch_mlp_frozen_seed999_best_beam5.json'),
    ('Cond B: resampler + Frozen', 'eval_test_condB_query_resampler_frozen_seed999_best_beam5.json'),
    ('Cond C: patch_mlp + LoRA', 'eval_test_condC_patch_mlp_lora_seed999_best_beam5.json'),
    ('Cond D: resampler + LoRA', 'eval_test_condD_query_resampler_lora_seed999_best_beam5.json'),
]

header = f"{'Condition (Seed ' + str(SEED) + ')':<34} | {'Prefix':>6} | {'Params':>10} | {'CIDEr':>7} | {'BLEU-1':>7} | {'BLEU-4':>7} | {'METEOR':>7} | {'ROUGE-L':>7} | {'ms/img':>8} | {'tok/s':>6}"
print(header)
print('-' * len(header))

for label, fname in runs:
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.isfile(fpath):
        print(f"{label:<34} | (not evaluated yet: {fname})")
        continue
    with open(fpath, 'r') as f:
        m = json.load(f)['metrics']
    print(
        f"{label:<34} | "
        f"{m.get('visual_prefix_length', 0):>6} | "
        f"{m.get('trainable_parameters', 0):>10,} | "
        f"{m.get('CIDEr', 0.0):>7.2f} | "
        f"{m.get('BLEU-1', 0.0):>7.2f} | "
        f"{m.get('BLEU-4', 0.0):>7.2f} | "
        f"{m.get('METEOR', 0.0):>7.2f} | "
        f"{m.get('ROUGE-L', 0.0):>7.2f} | "
        f"{m.get('latency_ms_per_image', 0.0):>8.1f} | "
        f"{m.get('tokens_per_second', 0.0):>6.1f}"
    )

Condition (Seed 999)               | Prefix |     Params |   CIDEr |  BLEU-1 |  BLEU-4 |  METEOR | ROUGE-L |   ms/img |  tok/s
------------------------------------------------------------------------------------------------------------------------------
Cond A: patch_mlp + Frozen         |     49 |  1,181,184 |    4.88 |   43.69 |    4.90 |   45.57 |   29.74 |    186.5 |   59.3
Cond B: resampler + Frozen         |     16 |    992,768 |    5.63 |   45.16 |    6.03 |   39.69 |   30.96 |    183.5 |   65.2
Cond C: patch_mlp + LoRA           |     49 |  1,476,096 |    5.83 |   43.62 |    5.69 |   42.62 |   30.35 |    236.4 |   50.7
Cond D: resampler + LoRA           |     16 |  1,287,680 |    5.87 |   45.03 |    5.75 |   42.53 |   30.47 |    208.2 |   55.1
